# Cython – Practice Exercises

Hands-on exercises following **`g_cython.ipynb`**. Each exercise has **one Markdown cell**
(goal, requirements, hints) and **one code cell** for you to implement, experiment, and time.

**Before you start**

- Work from the `training-py` repo root (or adjust paths). Use a dedicated folder such as
  **`cython_practice/`** for `.pyx` / `setup.py` files so you do not overwrite the guide’s
  `cython_build/` outputs.
- **Windows:** building extensions needs **MSVC** (Visual Studio Build Tools). From a normal
  Jupyter session, `cl.exe` may not be on `PATH`; see **§3 in `g_cython.ipynb`**
  (`vcvars64.bat`) or build from an **x64 Native Tools Command Prompt**.
- **Linux / macOS:** `gcc` / `clang` plus Python headers are enough.

**How to use each code cell**

- Fill in the `# TODO` sections.
- Run timings with `time.perf_counter()` and a fixed workload so comparisons are fair.
- When an exercise asks for a `.pyx`, write the file from the notebook (as in the guide) or
  edit it in your IDE.


### Contents

- [Exercise 1 – Baseline: micro-benchmark discipline](#exercise-1--baseline-micro-benchmark-discipline)
- [Exercise 2 – Typed loops: harmonic-style sum](#exercise-2--typed-loops-harmonic-style-sum)
- [Exercise 3 – `cpdef` / dual interface: moving average](#exercise-3--cpdef--dual-interface-moving-average)
- [Exercise 4 – `cdef class`: small order-book snapshot](#exercise-4--cdef-class-small-order-book-snapshot)
- [Exercise 5 – `nogil` and C-only helpers](#exercise-5--nogil-and-c-only-helpers)
- [Exercise 6 – Buffers: memoryview over bytes](#exercise-6--buffers-memoryview-over-bytes)
- [Exercise 7 – Fused types: one kernel, two dtypes](#exercise-7--fused-types-one-kernel-two-dtypes)
- [Exercise 8 – Build hygiene: `setup.py` and clean rebuild](#exercise-8--build-hygiene-setuppy-and-clean-rebuild)
- [Exercise 9 – Profile first, then optimize](#exercise-9--profile-first-then-optimize)


In [99]:
# Optional: shared imports and practice directory (edit if needed)

from __future__ import annotations

import os, sys, time, math, numpy
import shutil, subprocess, glob, importlib
from datetime import datetime, timedelta, timezone
from typing import Any, List, Dict, Callable, Tuple
from collections import deque, OrderedDict, defaultdict

from pandas import Series, DataFrame, Timestamp, Timedelta
from pandas import Index, DatetimeIndex, MultiIndex
from pandas import concat, date_range
from pathlib import Path

_notebook_dir = Path(".").resolve()
CYTHON_DIR = _notebook_dir / "cython"
CYTHON_DIR.mkdir(exist_ok = True)

try: import Cython; print("Cython:", Cython.__version__)
except ImportError: print("Install: pip install cython setuptools")
print("Practice dir:", CYTHON_DIR)

MSVC_DIR = r"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Auxiliary\Build\vcvars64.bat"

B, E = [1, 2, 5], range(3, 7)
RANGES = numpy.outer(B, numpy.pow(10, E)).T
RANGES = RANGES.reshape(-1).tolist()[: -2]

Cython: 3.1.2
Practice dir: C:\Users\GSL\Projects\Challenges\training-py\cython


In [100]:
class CySetup(OrderedDict):
    TAB = " " * 4
    PY_FILE = "setup.py"
    HEADER = str.join("\n", [
        "from Cython.Build import cythonize",
        "from setuptools import setup, Extension", "",
        "setup(ext_modules = cythonize(",
        TAB + "compiler_directives = {},",
        TAB + "module_list = [\n{}]", "))"
    ])
    CD_DEFAULT = {"language_level": "3"}
    BLOCK = TAB * 2 + "Extension(\"{}\", {})"
    def __init__(self, folder_path: str,
        msvc_path: str, cds: dict = None):

        self.folder: Path = Path(folder_path)
        self.pyfile = self.folder / self.PY_FILE
        if not cds: cds = self.CD_DEFAULT.copy()
        self.msvc_path, self.cds = msvc_path, cds

    def _assemble_str(self):
        blocks = list[str]()
        for item in self.items():
            blocks.append(self.BLOCK.format(*item))
        content = str.join(",\n", blocks)
        content = self.HEADER.format(self.cds, content)
        return content

    def write_to(self, name: str, code: str):
        self[name] = [name + ".pyx"]
        path: Path = self.folder / self[name][0]
        try: os.remove(path)
        except: pass
        path.write_text(code)
    
    def setup(self):
        content = self._assemble_str()
        try: os.remove(self.pyfile)
        except: pass
        self.pyfile.write_text(content)

    def build(self) -> subprocess.CompletedProcess:
        argv = [sys.executable, "setup.py", "build_ext", "--inplace"]
        cwd = str(self.folder)
        if (sys.platform != "win32"): return subprocess.run(
            argv, cwd = cwd, capture_output = True, text = True)
        oper = ( f'call "{self.msvc_path}" && cd /d "{cwd}" && '
            f'"{sys.executable}" setup.py build_ext --inplace')
        result = subprocess.run(oper, shell = True,
                text = True, capture_output = True)
        success = (result.returncode == 0)
        result_stdout = result.stdout if success else result.stderr
        result_stdout = result_stdout.strip("\n").strip(" ")
        return {"success": success, "output": result_stdout}

cysetup = CySetup(CYTHON_DIR, MSVC_DIR)

## Exercise 1 – Baseline: micro-benchmark discipline

**Goal:** Establish a **repeatable timing** habit before touching Cython.

**Scenario:** Implement **`fib_iter(n)`** — the *n*-th Fibonacci number using an **iterative**
algorithm (not recursion). State clearly your indexing (e.g. F(0)=0, F(1)=1).

**Requirements**

1. Implement `fib_iter(n: int) -> int` in pure Python.
2. Time it for at least two values of `n` (e.g. `10_000` and `50_000`) using `time.perf_counter()`.
3. Optionally add a **warm-up** call; briefly say why (or why not).

**Hints**

- For large `n`, Python’s arbitrary-precision `int` cost is part of the baseline story.
- Report **min** of a few runs or a **single** run — pick one and stick to it across exercises.


In [101]:
# Exercise 1 — your code here

def fib_iter(n: int) -> int:
    if (n <= 1): return 1.0
    past_2, past_1 = 0.0, 1.0
    delays = list[int]()
    start = time.perf_counter_ns()
    for _ in range(n):
        total = past_1 + past_2
        past_2, past_1 = past_1, total
        now = time.perf_counter_ns()
        delays.append(now - start)
    return total, numpy.array(delays)

# TODO: timing
result, delays = fib_iter(max(RANGES))
delays = delays[numpy.array(RANGES) - 1]
delays = Series(index = RANGES, data = delays)
delays.to_dict()

{1000: 77600,
 2000: 153400,
 5000: 369300,
 10000: 734100,
 20000: 1428600,
 50000: 3556800,
 100000: 7261600,
 200000: 15629000,
 500000: 40202700,
 1000000: 81564300}

## Exercise 2 – Typed loops: harmonic-style sum

**Goal:** Same “write `.pyx` + build” workflow as the guide, but a **different** kernel than
sum-of-squares.

**Scenario:** For `n >= 1`, let **H(n) = Σ(1/i)** for `i = 1..n` in **double precision**.

**Requirements**

1. `harmonic_py(n: int) -> float` in pure Python (baseline).
2. `harmonic_cy(n: int) -> float` in **`harmonic.pyx`** with `cdef double` accumulation and a typed index.
3. Minimal **`setup_harmonic.py`** with `Extension("harmonic", ["harmonic.pyx"])` + `cythonize`.
4. Build (`build_ext --inplace`), `import harmonic`, compare times for large `n` (e.g. `1_000_000`).

**Hints**

- On Windows, capture **stdout/stderr** if `subprocess.run` fails.
- See **`g_cython.ipynb` §2–3** for writing files and invoking the compiler.


In [102]:
# Exercise 2 — baseline + .pyx + build + timing

# TODO: harmonic_py
def harmonic_py(n: int):
    if (n <= 1): return n
    total = 0
    for i in range(1, n):
        total = total + 1/i
    return total
# TODO: write harmonic.pyx, setup_harmonic.py, subprocess build, import harmonic, compare
CODE = """
# distutils: language=c
# cython: language_level=3

def harmonic_cy(int n):
    if (n <= 1): return n
    cdef double total = 0
    cdef long i = 0
    for i in range(1, n):
        total = total + 1/i
    return total
"""

cysetup.write_to("harmonic", CODE)
cysetup.setup()
print(cysetup.build()["output"])

**********************************************************************
** Visual Studio 2026 Developer Command Prompt v18.4.3
** Copyright (c) 2026 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Compiling harmonic.pyx because it changed.
[1/1] Cythonizing harmonic.pyx
running build_ext
building 'harmonic' extension
"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -Ic:\DEV\Python\include -Ic:\DEV\Python\Include "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\include" "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.26100.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\

In [103]:
sys.path.insert(0, str(CYTHON_DIR))
import harmonic
importlib.reload(harmonic)
from harmonic import harmonic_cy

n = 10000
t = time.perf_counter_ns()
harmonic_py(n)
delay = time.perf_counter_ns() - t
print(f"Python :: \"harmonic_py({n})\" => delay = {delay} ns")
t = time.perf_counter_ns()
harmonic_cy(n)
delay = time.perf_counter_ns() - t
print(f"Cython :: \"harmonic_cy({n})\" => delay = {delay} ns")

Python :: "harmonic_py(10000)" => delay = 242800 ns
Cython :: "harmonic_cy(10000)" => delay = 29500 ns


## Exercise 3 – `cpdef` / dual interface: moving average

**Goal:** Practice **`cpdef`** (or a typed `def`) for code called **from Python** on hot paths.

**Scenario:** For `float` prices `p[0..n-1]` and window `k >= 1`, compute the **simple moving
average** at each `i >= k-1`: `mean(p[i-k+1 : i+1])`. For smaller `i`, use `nan` or skip — **document**.

**Requirements**

1. `moving_average_py(prices, k)`.
2. `moving_avg.pyx` with **`moving_average_cy`**, typed inner loop, `cpdef` if appropriate.
3. Correctness on a **tiny** hand-checked vector; then time on a long random list.

**Hints**

- **Bonus:** `O(n)` with a running sum instead of `O(n*k)`.
- Align with **§2–4** in `g_cython.ipynb`.


In [104]:
# TODO: moving_average_py + tiny test
def ma_py(series: list, k: int):
    if len(series) == 0: return series
    if not (k >= 1): raise ValueError(
      "Window size \"k\" must be >= 1")
    window = deque[float](maxlen = k)
    result = list[float]()
    total = 0.0

    for value in series:
        if (len(window) >= k):
            total -= window[0]
        window.append(value)
        total = total + value
        avg = total / len(window)
        result.append(avg)
        
    return result

# TODO: write ma.pyx, add to setup.py, subprocess build, import harmonic, compare
CODE = """
# distutils: language=c
# cython: language_level=3

cpdef list ma_cy(list prices, int k):
    cdef Py_ssize_t n = len(prices)
    cdef list result = []
    if (n == 0): return result
    if not (k >= 1): raise ValueError(
        "Window size k must be >= 1")
    cdef double total = 0.0
    cdef Py_ssize_t i

    for i in range(n):
        total = total + prices[i]
        if (i >= k): total -= prices[i - k]
        if (i < k - 1): result.append(float('nan'))
        else: result.append(total / k)

    return result
"""

cysetup.write_to("ma", CODE)
cysetup.setup()
print(cysetup.build()["output"])

**********************************************************************
** Visual Studio 2026 Developer Command Prompt v18.4.3
** Copyright (c) 2026 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Compiling ma.pyx because it changed.
[1/1] Cythonizing ma.pyx
running build_ext
building 'ma' extension
"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -Ic:\DEV\Python\include -Ic:\DEV\Python\Include "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\include" "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.26100.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\\shared" "-IC:\Pro

In [105]:
sys.path.insert(0, str(CYTHON_DIR))
import ma
importlib.reload(ma)
from ma import ma_cy

n = 100
series = numpy.arange(10000) / 400
series = numpy.sin(series * numpy.pi)
series = series.tolist()
t = time.perf_counter_ns()
result_py = ma_py(series, n)
delay = time.perf_counter_ns() - t
print(f"Python :: \"harmonic_py(series({len(series)}), {n})\" => delay = {delay} ns")
t = time.perf_counter_ns()
result_cy = ma_cy(series, n)
delay = time.perf_counter_ns() - t
print(f"Cython :: \"harmonic_cy(series({len(series)}), {n})\" => delay = {delay} ns")

Python :: "harmonic_py(series(10000), 100)" => delay = 873600 ns
Cython :: "harmonic_cy(series(10000), 100)" => delay = 364800 ns


## Exercise 4 – `cdef class`: small order-book snapshot

**Goal:** A **`cdef class`** distinct from the guide’s **message parser**: a **snapshot** of bid/ask **levels**.

**Scenario:** `OrderBookSnap` holds parallel bid/ask **arrays** (NumPy `float64` views or buffers you allocate).

- Methods: **`best_bid()`**, **`best_ask()`**, **`mid()`** (non-empty sides; define behavior if empty).

**Requirements**

1. Optional Python reference `OrderBookSnapPy`.
2. `order_book.pyx` with `cdef class OrderBookSnap`.
3. Build + one **hand-checked** example.

**Hints**

- **§4** in `g_cython.ipynb` — ownership of backing arrays vs memoryviews.


In [106]:
# Exercise 4

class TickPy:
    
    def __init__(self, symbol: str,
            pa: float, pb: float,
            va: float, vb: float,
            time: datetime = None):

        self.symbol = symbol
        if time is None: self.time = datetime.now(timezone.utc)
        self.pa, self.pb, self.va, self.vb = pa, pb, va, vb

    def __repr__(self):
        return (f"Tick({self.symbol} @ {self.time:%Y-%m-%d %H:%M:%S.%f} "
          f":: A: {self.pa:.6f}\\{self.va}, B: {self.pb:.6f}\\{self.vb})"
        )
    @property
    def pm(self):
        return (self.pa + self.pb) / 2
    @property
    def vwap(self):
        return (self.pa * self.va + self.pb * self.vb) / (self.va + self.vb)

# TODO: order_book.pyx, Extension, build, demo
CODE = """
# distutils: language=c
# cython: language_level=3

cdef class TickCy:
    cdef public double pa, pb, va, vb
    cdef public object symbol
    cdef public object time

    def __init__(self, symbol, double pa, double pb, double va, double vb, time=None):
        self.symbol = symbol
        self.pa = pa
        self.pb = pb
        self.va = va
        self.vb = vb
        if time is not None:
            self.time = time
        else:
            from datetime import datetime, timezone
            self.time = datetime.now(timezone.utc)

    cpdef double pm(self):
        return (self.pa + self.pb) / 2

    cpdef double vwap(self):
        return (self.pa * self.va + self.pb * self.vb) / (self.va + self.vb)

    def to_string(self):
        return f"Tick({self.symbol} @ {self.time:%Y-%m-%d %H:%M:%S.%f} :: A: {self.pa:.6f}\\\\{self.va}, B: {self.pb:.6f}\\\\{self.vb})"

    def __str__(self):
        return self.to_string()

    def __repr__(self):
        return self.to_string()
"""

cysetup.write_to("tick", CODE)
cysetup.setup()
print(cysetup.build()["output"])

**********************************************************************
** Visual Studio 2026 Developer Command Prompt v18.4.3
** Copyright (c) 2026 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Compiling tick.pyx because it changed.
[1/1] Cythonizing tick.pyx
running build_ext
building 'tick' extension
"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -Ic:\DEV\Python\include -Ic:\DEV\Python\Include "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.50.35717\include" "-IC:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.26100.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.26100.0\\shared" "-I

In [107]:
sys.path.insert(0, str(CYTHON_DIR))
import tick
importlib.reload(ma)
from tick import TickCy

tick_py = TickPy("EURUSD", 1.23456, 1.23455, 100, 50)
tick_cy = TickCy("EURUSD", 1.23456, 1.23455, 100, 50)
print("TickPy:", tick_py)
print("TickCy:", tick_cy)

delay_py = delay_cy = 0
for _ in range(n := 20000):
    t = time.perf_counter_ns()
    vwap_py = tick_py.vwap
    delay_py += time.perf_counter_ns() - t
    t = time.perf_counter_ns()
    vwap_cy = tick_cy.vwap
    delay_cy += time.perf_counter_ns() - t

print(f"Delay Python VWAP x {n}: {delay_py} ns")
print(f"Delay Cython VWAP x {n}: {delay_cy} ns")

TickPy: Tick(EURUSD @ 2026-04-09 05:33:34.478682 :: A: 1.234560\100, B: 1.234550\50)
TickCy: Tick(EURUSD @ 2026-04-09 05:33:34.478701 :: A: 1.234560\100.0, B: 1.234550\50.0)
Delay Python VWAP x 20000: 2869900 ns
Delay Cython VWAP x 20000: 2194800 ns


## Exercise 5 – `nogil` and C-only helpers

**Goal:** Split **nogil-safe** numeric work from Python bookkeeping (cf. **§4–5**, GIL discussion in the guide).

**Scenario:** **`pairwise_dot(x, y)`** = `Σ x[i]*y[i]` for equal-length float sequences.

- Implement **`cdef double pairwise_dot_core(...)`** with **`nogil`** using memoryviews / pointer+length.
- **`def pairwise_dot(x, y)`** validates, converts to contiguous `float64`, calls core, returns `float`.

**Requirements**

1. Correctness on length 3 by hand.
2. Time vs pure Python for length `500_000`.

**Hints**

- No Python C API inside `nogil` code.


In [ ]:
# Exercise 5

def dot_py_1(x: list[float], y: list[float]):
    if (len(x) != len(y)): raise ValueError("Inputs have different length.")
    return sum(float(x_i) * float(y_i) for x_i, y_i in zip(x, y))

def dot_py_2(x: list[float], y: list[float]):
    if (len(x) != len(y)): raise ValueError("Inputs have different length.")
    # Convert to contiguous float64 arrays
    x_i = numpy.ascontiguousarray(x, dtype = numpy.float64)
    y_i = numpy.ascontiguousarray(y, dtype = numpy.float64)
    return numpy.dot(x_i, y_i)

CODE = """
# distutils: language=c
# cython: language_level=3

def dot_cy(double[::1] x, double[::1] y):
    cdef Py_ssize_t n = x.shape[0]
    cdef Py_ssize_t i
    cdef double acc = 0
    for i in range(n):
        acc += x[i] * y[i]
    return acc
    
"""

cysetup.write_to("dot", CODE)
cysetup.setup()
print(cysetup.build()["output"])

## Exercise 6 – Buffers: memoryview over bytes

**Goal:** Typed iteration over **`bytes`** — typical for **wire-format** style parsing (different from the guide’s streaming study).

**Scenario:** Interpret `data: bytes` as little-endian **uint32** words. Implement **`sum_le_u32(data) -> int`**
with `len(data) % 4 == 0` (else `ValueError`).

**Requirements**

1. Slow baseline using `int.from_bytes` per word.
2. Cython: tight loop over a memoryview (no list of Python ints).
3. Equality check on random payload, length `4 * 200_000`.

**Hints**

- Sum may exceed 32 bits — widen accumulator (`unsigned long long` or Python `int` at the end).


In [ ]:
# Exercise 6

# TODO: sum_le_u32_py

# TODO: sum_le_u32.pyx + build + tests + timing


## Exercise 7 – Fused types: one kernel, two dtypes

**Goal:** **`ctypedef fused`** (or equivalent) so **one** implementation serves **`int`** and **`double`**
1-D contiguous data.

**Scenario:** **`argmax_1d(a)`** → index of maximum element. **Define tie-breaking** (first vs last max).

**Requirements**

1. One `.pyx` module; tests with `np.arange` (integer) and `np.linspace` (float).

**Hints**

- Cython docs: **Fused types**. Fallback: two `cpdef` functions if fused types fight you — note the trade-off.


In [ ]:
# Exercise 7

import numpy as np

# TODO: argmax_fused.pyx + build + tests


## Exercise 8 – Build hygiene: `setup.py` and clean rebuild

**Goal:** Reliable **edit → compile → import** without stale extensions.

**Requirements**

1. Pick one practice module (e.g. `harmonic`).
2. Change a **`VERSION`** string constant in the `.pyx`, rebuild, confirm via `import` / `reload`.
3. Describe **one** clean strategy: delete `build/`, remove `*.c`, or `setup.py clean`.

**Hints**

- **`importlib.reload`** or restart kernel — **`g_cython.ipynb` §3**.
- Ignore generated artifacts in git if applicable.


In [ ]:
# Exercise 8

import importlib

# TODO: demonstrate version bump + reload

# import harmonic
# importlib.reload(harmonic)


## Exercise 9 – Profile first, then optimize

**Goal:** Tie **profiling** to a **targeted** Cython or algorithmic fix (without repeating the guide’s WebSocket case).

**Scenario:** **`rolling_std_py(prices, window)`** — rolling standard deviation (choose **sample** vs **population**
stdev and **document**). Start with a **clear, slow** nested-loop implementation.

**Requirements**

1. Run **`cProfile`** on non-trivial `n` and `window`; locate the hotspot.
2. Improve with **Cython** and/or **`O(n)`** rolling sum of squares (Welford optional stretch goal).
3. Print a short **before/after** timing table.

**Hints**

- `pstats.SortKey.TIME` or sort by cumulative time.


In [ ]:
# Exercise 9

import cProfile
import pstats

import numpy as np

# TODO: rolling_std_py + cProfile

# TODO: fast path + compare
